# Tugas Praktikum JS02 - Dataset PBL


## Pra-pengolahan Dataset Berita Hoaks dan Valid


### Load data hoaks dan valid


In [1]:
import pandas as pd
import numpy as np

# Baca dataset hoaks Komdigi dan dataset berita valid
df_hoaks = pd.read_csv('data/komdigi_hoaks.csv')
df_valid = pd.read_csv('data/berita_valid_terbaru.csv')

print('Dimensi dataset hoaks Komdigi:', df_hoaks.shape)
print('Dimensi dataset berita valid :', df_valid.shape)


Dimensi dataset hoaks Komdigi: (4176, 13)
Dimensi dataset berita valid : (749, 7)


### Cek kolom dan missing value


In [2]:
print('Kolom dataset hoaks:', df_hoaks.columns.tolist())
print('Kolom dataset valid:', df_valid.columns.tolist())

print('\nMissing value pada dataset hoaks:')
print(df_hoaks[['title', 'excerpt', 'body_text']].isnull().sum())

print('\nMissing value pada dataset valid:')
print(df_valid[['title', 'excerpt']].isnull().sum())


Kolom dataset hoaks: ['id', 'url', 'title', 'slug', 'published_at', 'view_count', 'excerpt', 'body_html', 'body_text', 'main_image_url', 'category', 'tags', 'topics']
Kolom dataset valid: ['id', 'source', 'title', 'excerpt', 'url', 'published_at', 'label']

Missing value pada dataset hoaks:
title           0
excerpt      2883
body_text       0
dtype: int64

Missing value pada dataset valid:
title      0
excerpt    0
dtype: int64


### Pembersihan teks


In [3]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ''
    # Hapus tag prefix [HOAKS] atau (DISINFORMASI) agar tidak terjadi data leakage
    text = re.sub(r'\[(HOAKS|DISINFORMASI|MISINFORMASI|KLARIFIKASI|SALAH|FALSE)\]', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\((HOAKS|DISINFORMASI|MISINFORMASI|KLARIFIKASI|SALAH|FALSE)\)', '', text, flags=re.IGNORECASE)
    # Hapus URL
    text = re.sub(r'https?:\/\/\S+|www\.\S+', '', text)
    # Hapus tag html
    text = re.sub(r'<.*?>', '', text)
    # Hapus karakter selain huruf dan spasi
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # Rapikan spasi berlebih
    return re.sub(r'\s+', ' ', text.lower()).strip()

df_hoaks['teks_bersih'] = (df_hoaks['title'].fillna('') + ' ' + df_hoaks['excerpt'].fillna('')).apply(clean_text)
df_valid['teks_bersih'] = (df_valid['title'].fillna('') + ' ' + df_valid['excerpt'].fillna('')).apply(clean_text)

print('Contoh teks bersih hoaks:')
print(df_hoaks['teks_bersih'].iloc[0][:150])
print('\nContoh teks bersih valid:')
print(df_valid['teks_bersih'].iloc[0][:150])


Contoh teks bersih hoaks:
tautan pencairan bsu september sebesar rp kepala biro humas kementerian ketenagakerjaan kemnaker faried abdurrahman nur yuliono mengatakan belum ada k

Contoh teks bersih valid:
terpopuler kado hari guru hingga riset vaksin malaria sejumlah berita unggulan rabu untuk disimak bocorkan kado hari guru tidak ada phk guru non asn h


### Labeling dan balance data


In [4]:
# Encoding label target: 1 = Hoaks, 0 = Valid
df_hoaks['label'] = 1
df_valid['label'] = 0

# Penyeimbangan data (Balanced Dataset 1:1)
df_hoaks_sampled = df_hoaks.sample(n=len(df_valid), random_state=42)

# Menggabungkan data hoaks dan valid
df_pbl = pd.concat([
    df_hoaks_sampled[['teks_bersih', 'label']],
    df_valid[['teks_bersih', 'label']]
], ignore_index=True)

# Filter teks yang terlalu pendek (di bawah 10 karakter)
df_pbl = df_pbl[df_pbl['teks_bersih'].str.len() > 10].reset_index(drop=True)

print('Dimensi data gabungan PBL:', df_pbl.shape)
print('\nDistribusi label target:')
print(df_pbl['label'].value_counts())
df_pbl.head()


Dimensi data gabungan PBL: (1498, 2)

Distribusi label target:
label
1    749
0    749
Name: count, dtype: int64


,teks_bersih,label
0,presiden prabowo minta maaf ikut bop akan kemb...,1
1,video presiden prabowo akan membagikan bantuan...,1
2,pemerintah kota tangerang selatan akan legalka...,1
3,tautan rekrutmen cpns kementerian imigrasi,1
4,ramuan rebusan jahe kunyit kencur dan serai be...,1


### Split data train test


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df_pbl['teks_bersih'],
    df_pbl['label'],
    test_size=0.2,
    random_state=42,
    stratify=df_pbl['label']
)

print(f'Jumlah data latih: {len(X_train)} baris')
print(f'Jumlah data uji  : {len(X_test)} baris')


Jumlah data latih: 1198 baris
Jumlah data uji  : 300 baris


### Ekstraksi fitur TF-IDF


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

print('Dimensi matriks fitur train:', X_train_vec.shape)
print('Dimensi matriks fitur test :', X_test_vec.shape)
print('\nContoh 15 fitur kosakata:')
print(tfidf.get_feature_names_out()[:15])


Dimensi matriks fitur train: (1198, 1000)
Dimensi matriks fitur test : (300, 1000)

Contoh 15 fitur kosakata:
['abdul' 'aceh' 'ada' 'adalah' 'adanya' 'administrasi' 'agama' 'agar'
 'agustus' 'ahmad' 'ai' 'air' 'ajak' 'ajang' 'akan']


### Simpan data gabungan bersih untuk jobsheet berikutnya


In [7]:
# Simpan dataset hasil pra-pengolahan
df_pbl.to_csv('data/pbl_hoaks_valid_clean.csv', index=False)
print('Dataset gabungan bersih tersimpan di data/pbl_hoaks_valid_clean.csv')
print('Jumlah baris:', len(df_pbl))


Dataset gabungan bersih tersimpan di data/pbl_hoaks_valid_clean.csv
Jumlah baris: 1498
